In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix

In [5]:
import torch
import torch.nn as nn

class CIFAR10AlexNet(nn.Module):
    def __init__(self, num_classes=10):
        super(CIFAR10AlexNet, self).__init__()
        self.features = nn.Sequential(
            # 32x32 입력에 맞게 커널 크기 3, stride 1로 조정
            nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2), # 16x16
            
            nn.Conv2d(64, 192, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2), # 8x8
            
            nn.Conv2d(192, 384, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            
            nn.Conv2d(384, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2), # 4x4
        )
        self.classifier = nn.Sequential(
            nn.Dropout(),
            nn.Linear(256 * 4 * 4, 4096),
            nn.ReLU(inplace=True),
            nn.Dropout(),
            nn.Linear(4096, 4096),
            nn.ReLU(inplace=True),
            nn.Linear(4096, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

#################################################
# 0. 디바이스 설정
#################################################
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

#################################################
# 1. CIFAR-10 데이터셋 전처리 및 로드
#################################################
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

full_train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

# 45,000개 학습용, 5,000개 검증용 분할
train_size = 45000
val_size = 5000
train_dataset, val_dataset = random_split(full_train_dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)


# 모델 선언 시 변경된 클래스 사용
model = CIFAR10AlexNet(num_classes=10).to(device)

#################################################
# 3. 손실 함수와 옵티마이저 설정
#################################################
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

#################################################
# 4. 학습 및 검증 루프 (Training & Validation Loop)
#################################################
epochs = 10
for epoch in range(epochs):
    # --- [Training] ---
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        train_total += labels.size(0)
        train_correct += predicted.eq(labels).sum().item()
        
    train_epoch_loss = train_loss / train_total
    train_epoch_acc = train_correct / train_total

    # --- [Validation] ---
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()
            
    val_epoch_loss = val_loss / val_total
    val_epoch_acc = val_correct / val_total
    
    print(f"Epoch [{epoch+1}/{epochs}] | "
          f"Train Loss: {train_epoch_loss:.4f}, Train Acc: {train_epoch_acc*100:.2f}% | "
          f"Val Loss: {val_epoch_loss:.4f}, Val Acc: {val_epoch_acc*100:.2f}%")

#################################################
# 5. 테스트 평가 (Test Evaluation)
#################################################
model.eval()
test_loss, test_correct, test_total = 0.0, 0, 0
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        test_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        test_total += labels.size(0)
        test_correct += predicted.eq(labels).sum().item()

test_epoch_loss = test_loss / test_total
test_epoch_acc = test_correct / test_total
print(f"\n[Test Result] Loss: {test_epoch_loss:.4f}, Accuracy: {test_epoch_acc*100:.2f}%")

Using device: cpu


100%|██████████| 170M/170M [27:25<00:00, 104kB/s]  


Epoch [1/10] | Train Loss: 1.6667, Train Acc: 36.62% | Val Loss: 1.2978, Val Acc: 51.74%
Epoch [2/10] | Train Loss: 1.2222, Train Acc: 56.06% | Val Loss: 1.0524, Val Acc: 62.30%
Epoch [3/10] | Train Loss: 1.0258, Train Acc: 63.18% | Val Loss: 0.9712, Val Acc: 65.02%
Epoch [4/10] | Train Loss: 0.8961, Train Acc: 68.54% | Val Loss: 0.8683, Val Acc: 69.76%
Epoch [5/10] | Train Loss: 0.7862, Train Acc: 72.36% | Val Loss: 0.8397, Val Acc: 70.66%
Epoch [6/10] | Train Loss: 0.7062, Train Acc: 75.47% | Val Loss: 0.7279, Val Acc: 74.58%
Epoch [7/10] | Train Loss: 0.6363, Train Acc: 77.88% | Val Loss: 0.7564, Val Acc: 74.38%
Epoch [8/10] | Train Loss: 0.5775, Train Acc: 79.84% | Val Loss: 0.7160, Val Acc: 76.62%
Epoch [9/10] | Train Loss: 0.5242, Train Acc: 81.56% | Val Loss: 0.7537, Val Acc: 75.56%
Epoch [10/10] | Train Loss: 0.4792, Train Acc: 83.51% | Val Loss: 0.7912, Val Acc: 74.38%

[Test Result] Loss: 0.8078, Accuracy: 75.40%
